# scRNA-seq walkthrough

An interactive, step-by-step version of `scrna_pipeline.py` on the bundled
example dataset (3k PBMCs). Plots render inline. For batch / reproducible runs
on any dataset, use the script + `config/config.yaml` instead.


In [ ]:
import scanpy as sc
import numpy as np

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor='white')
np.random.seed(0)


## 1. Load data
Here we use the example PBMC dataset. To use your own, load a 10x/`.h5ad` file
with `sc.read_10x_mtx(...)`, `sc.read_10x_h5(...)`, or `sc.read_h5ad(...)`.


In [ ]:
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()
adata


## 2. Quality control (MAD-based)
Compute QC metrics, then flag outliers by median absolute deviation.


In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=[20], log1p=True, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts','total_counts','pct_counts_mt'], jitter=0.4, multi_panel=True)


In [ ]:
def is_outlier(a, metric, nmads=5):
    M = a.obs[metric]; med = np.median(M); mad = np.median(np.abs(M - med))
    return (M < med - nmads*mad) | (M > med + nmads*mad)

adata.obs['outlier'] = (is_outlier(adata,'log1p_total_counts') |
                        is_outlier(adata,'log1p_n_genes_by_counts') |
                        is_outlier(adata,'pct_counts_in_top_20_genes'))
adata.obs['mt_outlier'] = is_outlier(adata,'pct_counts_mt',3)
print('cells before:', adata.n_obs)
adata = adata[~(adata.obs['outlier'] | adata.obs['mt_outlier'])].copy()
sc.pp.filter_cells(adata, min_genes=200); sc.pp.filter_genes(adata, min_cells=3)
print('cells after: ', adata.n_obs)


## 3. Normalize, log, highly variable genes


In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pl.highly_variable_genes(adata)
adata = adata[:, adata.var['highly_variable']].copy()


## 4. Scale + PCA


In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=50, svd_solver='arpack', random_state=0)
sc.pl.pca_variance_ratio(adata, n_pcs=50)


## 5. Neighbors, UMAP, Leiden clustering


In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.tl.leiden(adata, resolution=1.0, random_state=0, flavor='igraph', n_iterations=2, directed=False)
sc.pl.umap(adata, color='leiden', legend_loc='on data')


## 6. Marker genes per cluster


In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon', use_raw=True)
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)
sc.get.rank_genes_groups_df(adata, group=None).head()


## Next steps
- Change parameters in `config/config.yaml` and run `python scrna_pipeline.py` for a reproducible batch run.
- Annotate clusters using the marker genes above.
- Swap in your own dataset via the `data` block in the config (GEO / 10x / h5ad / URL).
